# CDC/ATSDR Social Vulnerability Index: the reused-name trap

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cstirry/field-match/blob/main/examples/cdc_atsdr_svi.ipynb)

Crosswalks two releases of the [Social Vulnerability Index](https://www.atsdr.cdc.gov/place-health/php/svi/svi-data-documentation-download.html), 2010 vs. 2022 county files.

Most of the 103 columns were renamed between releases. Two were reused with different meanings: in 2010, `ST` held text state abbreviations and `STATE` held numeric codes; by 2022 the numeric code was called `ST`. A name-based merge would silently swap abbreviations for codes. field-match checks the contents as well as the names, catches it, and reports both columns as suspect.

In [ ]:
%pip install -q field-match

## 1. Get the data

Downloads once into `data/`; reused on later runs.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

from field_match import compare

DATA_DIR = Path("data")

FILES = {
    "SVI_2010_US_county.csv": "https://svi.cdc.gov/Documents/Data/2010/csv/states_counties/SVI_2010_US_county.csv",
    "SVI_2022_US_county.csv": "https://svi.cdc.gov/Documents/Data/2022/csv/states_counties/SVI_2022_US_county.csv",
}


def fetch(filename: str) -> Path:
    path = DATA_DIR / filename
    if not path.exists():
        DATA_DIR.mkdir(exist_ok=True)
        print(f"Downloading {filename} ...")
        urlretrieve(FILES[filename], path)
    return path


svi_2010 = fetch("SVI_2010_US_county.csv")
svi_2022 = fetch("SVI_2022_US_county.csv")

## 2. Compare the two releases

`compare()` accepts file paths directly. Takes about 15 seconds.

In [2]:
report = compare(svi_2010, svi_2022)
print(report)

field-match comparison: 103 reference columns vs 158 new columns
(3,143 vs 3,144 rows; match_threshold 0.50, verified_threshold 0.75)

  16  verified  column name and contents both match
  64  renamed   contents match, but under a different column name
   4  suspect   column name matches, but contents do not
  21  dropped   missing from the new dataset
  75  added     missing from the reference dataset

Use report.summary() to list the columns in each category.


## 3. Why each suspect column was flagged

This is where the reused `ST`/`STATE` names surface. The contents give it away, so field-match refuses to pair them by name alone.

In [3]:
for s in report.suspect:
    print(f"{s.name}: {s.reason}")

M_HU: matched its namesake, but the contents drifted (score 0.73 is below verified_threshold 0.75)
LOCATION: matched its namesake, but the contents drifted (score 0.72 is below verified_threshold 0.75)
ST: contents are different types (text in the reference, numeric in the new data); the reference column matched 'ST_ABBR' instead; the new column matched from 'STATE' instead
STATE: contents are different types (numeric in the reference, text in the new data); the reference column matched 'ST' instead


## 4. Drill into one column's candidates

Not sure about a match? `report.candidates()` returns the ranked evidence as a DataFrame.

In [4]:
report.candidates("E_POV")

,source,target,family,name_score,content_score,score
0,E_POV,E_POV150,numeric,0.9,0.829,0.857
1,E_POV,E_DISABL,numeric,0.5,0.953,0.772
2,E_POV,E_AGE65,numeric,0.5,0.899,0.739
3,E_POV,E_MINRTY,numeric,0.5,0.879,0.728
4,E_POV,E_AGE17,numeric,0.5,0.859,0.716


## 5. Apply or review the mapping

Happy with it? Apply directly:

```python
import pandas as pd
aligned_2022 = report.apply(pd.read_csv(svi_2022))
```

Prefer to hand-check first, `report.rename_snippet()` writes a pasteable dict with one commented line per decision:

In [5]:
print(report.rename_snippet())

# Matched with the same name in both files - no rename needed:
#   'FIPS'  (score=1.00, numeric)
#   'LOCATION'  (score=0.72, text)
#   'E_TOTPOP'  (score=0.99, numeric)
#   'M_TOTPOP'  (score=1.00, numeric)
#   'E_HU'  (score=0.98, numeric)
#   'M_HU'  (score=0.73, numeric)
#   'E_UNEMP'  (score=0.91, numeric)
#   'M_UNEMP'  (score=0.95, numeric)
#   'E_LIMENG'  (score=0.98, numeric)
#   'M_LIMENG'  (score=0.82, numeric)
#   'E_MUNIT'  (score=0.97, numeric)
#   'M_MUNIT'  (score=0.87, numeric)
#   'E_MOBILE'  (score=0.98, numeric)
#   'M_MOBILE'  (score=0.96, numeric)
#   'E_CROWD'  (score=0.98, numeric)
#   'M_CROWD'  (score=0.82, numeric)
#   'E_NOVEH'  (score=0.98, numeric)
#   'M_NOVEH'  (score=0.96, numeric)
rename_dict = {
    'ST': 'STATE',  # score=0.92 (numeric)
    'ST_ABBR': 'ST',  # score=0.87 (text)
    'E_HH': 'HH',  # score=0.87 (numeric)
    'E_POV150': 'E_POV',  # score=0.86 (numeric)
    'M_POV150': 'M_POV',  # score=0.91 (numeric)
    'E_HBURD': 'HU',  # score=0.52 